# Урок 20. Условные функции и статистика в таблицах

9 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 19](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-19.ipynb) · [Урок 21 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-21.ipynb)

---

СУММЕСЛИ, СЧЁТЕСЛИ, СРЗНАЧЕСЛИ, вложенные ЕСЛИ. Расчёт долей и процентов. Проверка расчётов программой.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-20", name=ФИО, klass=Класс)

## Разбираемся

### Считать не всё, а нужное

`СУММ` и `СРЗНАЧ` считают весь диапазон целиком. Но настоящие вопросы
звучат иначе: сколько **пятёрок** в 9А, какая средняя цена
**у книг дороже 500**, сколько заказов **из Москвы**.

Это условные функции, и их всего три плюс одна.

### Три функции с условием

| Функция | Что считает |
|---|---|
| `=СЧЁТЕСЛИ(диапазон; условие)` | сколько ячеек подходит |
| `=СУММЕСЛИ(диапазон; условие; суммируемый)` | сумму подходящих |
| `=СРЗНАЧЕСЛИ(диапазон; условие; усредняемый)` | среднее подходящих |

Условие пишут либо значением, либо **строкой со знаком**:

```
=СЧЁТЕСЛИ(B2:B100; 5)          пятёрок
=СЧЁТЕСЛИ(B2:B100; ">3")       больше трёх
=СЧЁТЕСЛИ(A2:A100; "9А")       из девятого А
=СЧЁТЕСЛИ(A2:A100; "Ив*")      фамилии на «Ив»
```

Знаки сравнения и звёздочку берут **в кавычки** — иначе таблица
не поймёт, где кончается условие. Это ловушка номер один.

### Третий аргумент

У `СУММЕСЛИ` три аргумента, и их путают чаще всего:

```
=СУММЕСЛИ(A2:A100; "9А"; C2:C100)
           ↑          ↑     ↑
     где искать   что искать  что сложить
```

«Пройди по столбцу A, найди строки с 9А, сложи из них столбец C».
Если третий аргумент не указать, сложится сам столбец A.

### Условие по нескольким столбцам

Когда условий несколько, к имени добавляется `МН` — «множественное»:

```
=СЧЁТЕСЛИМН(A2:A100; "9А"; B2:B100; 5)
```

Читается: «строк, где класс 9А **и** оценка 5». Условия соединяются
только через «и»; для «или» складывают два СЧЁТЕСЛИМН.

### Вложенные ЕСЛИ

Когда исходов больше двух, `ЕСЛИ` вкладывают друг в друга:

```
=ЕСЛИ(A1>=85; "5"; ЕСЛИ(A1>=70; "4"; ЕСЛИ(A1>=50; "3"; "2")))
```

Это ровно `if / elif / elif / else` из Python, записанный в строку.
И у него та же ловушка: **порядок условий**. Начнёшь с `A1>=50` —
и девяносто баллов получат тройку, потому что первое условие
подошло раньше.

Правило то же, что в программировании: от самого строгого условия
к самому мягкому.

### Доли и проценты

Доля — это часть, делённая на целое:

```
=C2/$C$10          доля строки в итоге
```

Знаменатель закрепляют долларами, иначе при копировании формула
начнёт делить на что попало.

Чтобы получить проценты, не нужно умножать на 100: достаточно
применить к ячейке процентный формат, и `0,25` покажется как `25 %`.
Умножение на 100 **и** процентный формат дадут 2500 % — ловушка
номер два.

### Зачем проверять расчёт кодом

Формулу в таблице легко растянуть не на тот диапазон, и ошибка
ничем себя не выдаёт: число есть, выглядит правдоподобно.

Поэтому серьёзный расчёт проверяют **вторым способом**: та же
статистика, посчитанная программой, должна совпасть с таблицей.
Не совпало — ищем, кто из двоих врёт. Этим и займёмся.

## Смотрим, как это работает

### Пример 1. Данные для расчётов

Небольшой журнал: класс, фамилия, балл.

In [ ]:
журнал = [
    ["9А", "Иванов", 92],
    ["9А", "Петрова", 78],
    ["9А", "Сидоров", 45],
    ["9Б", "Кузнецова", 88],
    ["9Б", "Орлов", 61],
    ["9Б", "Никитина", 95],
    ["9В", "Жуков", 53],
    ["9В", "Ильина", 72],
]

for строка in журнал:
    print(f"{строка[0]}  {строка[1]:<12} {строка[2]}")

### Пример 2. СЧЁТЕСЛИ

In [ ]:
сколько_9а = 0
for класс, фамилия, балл in журнал:
    if класс == "9А":
        сколько_9а += 1

сколько_выше_70 = 0
for класс, фамилия, балл in журнал:
    if балл > 70:
        сколько_выше_70 += 1

print("Учеников 9А:", сколько_9а, '— в таблице: =СЧЁТЕСЛИ(A2:A9; "9А")')
print("Баллов выше 70:", сколько_выше_70, '— в таблице: =СЧЁТЕСЛИ(C2:C9; ">70")')

### Пример 3. СУММЕСЛИ и СРЗНАЧЕСЛИ

In [ ]:
сумма_9б = 0
сколько_9б = 0

for класс, фамилия, балл in журнал:
    if класс == "9Б":
        сумма_9б += балл
        сколько_9б += 1

print("Сумма баллов 9Б:", сумма_9б)
print("Средний балл 9Б:", сумма_9б / сколько_9б)
print()
print('В таблице: =СУММЕСЛИ(A2:A9; "9Б"; C2:C9)')
print('           =СРЗНАЧЕСЛИ(A2:A9; "9Б"; C2:C9)')

Обратите внимание: счётчик `сколько_9б` нужен и здесь. Делить
на общее количество учеников нельзя — это ошибка, из-за которой
средний балл класса оказывается втрое меньше настоящего.

### Пример 4. Два условия сразу

In [ ]:
отличников_9б = 0
for класс, фамилия, балл in журнал:
    if класс == "9Б" and балл >= 85:
        отличников_9б += 1

print("В 9Б набрали 85 и больше:", отличников_9б)
print('В таблице: =СЧЁТЕСЛИМН(A2:A9; "9Б"; C2:C9; ">=85")')

### Пример 5. Вложенные ЕСЛИ

In [ ]:
def оценка(балл):
    if балл >= 85:
        return 5
    elif балл >= 70:
        return 4
    elif балл >= 50:
        return 3
    else:
        return 2


for класс, фамилия, балл in журнал:
    print(f"{фамилия:<12} {балл:>3} → {оценка(балл)}")

А вот что бывает, если перепутать порядок условий.

In [ ]:
def неправильно(балл):
    if балл >= 50:
        return 3
    elif балл >= 70:
        return 4
    elif балл >= 85:
        return 5
    return 2


print("Было 92 балла, а оценка:", неправильно(92))
print("Первое же условие подошло — до пятёрки дело не дошло.")

### Пример 6. Доли и проценты

In [ ]:
всего = 0
for класс, фамилия, балл in журнал:
    всего += балл

print("Сумма всех баллов:", всего)
print()
for класс, фамилия, балл in журнал:
    доля = балл / всего
    print(f"{фамилия:<12} {балл:>3}  доля {доля:.3f}  = {доля * 100:.1f} %")

В таблице это формула `=C2/$C$10` с процентным форматом ячейки.
Доллары держат знаменатель на месте при копировании вниз.

### Пример 7. Проверка таблицы кодом

Допустим, в таблице посчитали средний балл и получили 73,0.
Проверим независимо.

In [ ]:
из_таблицы = 73.0

баллы = [строка[2] for строка in журнал]
из_кода = sum(баллы) / len(баллы)

print("Таблица говорит:", из_таблицы)
print("Код говорит:    ", round(из_кода, 3))
print("Совпало:", abs(из_таблицы - из_кода) < 0.001)

Не совпало — значит, в таблице диапазон захватил лишнюю строку
или потерял нужную. Это первое, что нужно проверить.

## Пробуем сами

Во всех задачах журнал приходит списком строк вида
`[класс, фамилия, балл]`.

### Задача 1. СЧЁТЕСЛИ

Функция возвращает количество учеников заданного класса.

In [ ]:
def счётесли_класс(журнал, класс):
    return ...

In [ ]:
si.check("1", счётесли_класс, [
    (([["9А", "И", 90], ["9Б", "П", 80], ["9А", "С", 70]], "9А"), 2),
    (([["9А", "И", 90]], "9Б"), 0),
    (([], "9А"), 0),
])

### Задача 2. СУММЕСЛИ

Функция возвращает сумму баллов учеников заданного класса.

In [ ]:
def суммесли_класс(журнал, класс):
    return ...

In [ ]:
si.check("2", суммесли_класс, [
    (([["9А", "И", 90], ["9Б", "П", 80], ["9А", "С", 70]], "9А"), 160),
    (([["9А", "И", 90]], "9Б"), 0),
    (([], "9А"), 0),
])

### Задача 3. СРЗНАЧЕСЛИ

Функция возвращает средний балл заданного класса. Делить нужно
на количество учеников **этого класса**, а не всех.

Если в классе никого нет, верните 0 — в таблице на этом месте
было бы `#ДЕЛ/0!`.

In [ ]:
def срзначесли_класс(журнал, класс):
    return ...

In [ ]:
si.check("3", срзначесли_класс, [
    (([["9А", "И", 90], ["9Б", "П", 80], ["9А", "С", 70]], "9А"), 80.0),
    (([["9А", "И", 90], ["9Б", "П", 80]], "9Б"), 80.0),
    (([["9А", "И", 90]], "9В"), 0),
])

### Задача 4. СЧЁТЕСЛИМН

Функция возвращает количество учеников заданного класса,
у которых балл не меньше порога.

In [ ]:
def счётеслимн(журнал, класс, порог):
    return ...

In [ ]:
si.check("4", счётеслимн, [
    (([["9А", "И", 90], ["9А", "П", 60], ["9Б", "С", 95]], "9А", 85), 1),
    (([["9А", "И", 90], ["9А", "П", 60]], "9А", 50), 2),
    (([["9А", "И", 90]], "9Б", 10), 0),
])

### Задача 5. Вложенные ЕСЛИ

Функция переводит балл в оценку: 85 и выше — 5, от 70 — 4,
от 50 — 3, ниже — 2.

Следите за порядком условий.

In [ ]:
def в_оценку(балл):
    return ...

In [ ]:
si.check("5", в_оценку, [
    (92, 5),
    (85, 5),
    (70, 4),
    (50, 3),
    (49, 2),
    (0, 2),
])

### Задача 6. Доля в процентах

Функция получает часть и целое и возвращает долю в процентах,
округлённую до одного знака после запятой.

Если целое равно нулю, верните 0.

In [ ]:
def процент(часть, целое):
    return ...

In [ ]:
si.check("6", процент, [
    ((7, 28), 25.0),
    ((1, 3), 33.3),
    ((5, 5), 100.0),
    ((0, 10), 0.0),
    ((3, 0), 0),
])

## Домашнее задание

### Домашнее задание 1. Статистика в таблице и в коде

Возьмите таблицу из прошлого урока и посчитайте в ней формулами:

* количество значений выше среднего;
* сумму и среднее по одной категории (класс, город, тип товара);
* долю каждой категории в процентах от итога.

Затем повторите те же три расчёта программой и сверьте.
Если числа разошлись — найдите, кто ошибся. Покажите оба
результата учителю.

### Домашнее задание 2. Сколько подходит под два условия

Функция возвращает количество строк, где класс совпадает
**и** балл строго меньше порога.

In [ ]:
def слабых(журнал, класс, порог):
    return ...

In [ ]:
si.check("дз2", слабых, [
    (([["9А", "И", 40], ["9А", "П", 80], ["9Б", "С", 30]], "9А", 50), 1),
    (([["9А", "И", 40], ["9А", "П", 30]], "9А", 50), 2),
    (([["9А", "И", 90]], "9А", 50), 0),
])

### Домашнее задание 3. Сколько это процентов

Сколько процентов составляет 7 от 28?

In [ ]:
#@title 🏠 Домашнее 3. Проценты { display-mode: "form" }
#@markdown Впиши число процентов
процентов = 0 #@param {type:"integer"}

si.ответ("дз3", процентов, "b7a56873cd771f2c",
         hint="Раздели и умножь на сто.")

---

### Любопытно

Проценты умеют путать даже взрослых. Заголовок «доля отказов выросла
с 2 % до 4 %» можно честно написать двумя способами: «выросла
на 2 процентных пункта» и «выросла на 100 %». Оба верны, но второй
звучит как катастрофа.

Поэтому в статистике различают **процент** и **процентный пункт**,
а в новостях — почти никогда. Теперь вы умеете проверять такие
заголовки: делите одно на другое и смотрите, о чём речь.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 19](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-19.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 21 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-21.ipynb)